In [ ]:
# This cell installs all required libraries for the V100 environment.
# Using -q for a quieter installation output.
!pip install -q \
  "transformers @ git+https://github.com/huggingface/transformers" \
  "torch==2.3.1" \
  "torchvision==0.18.1" \
  "torchcodec==0.2.1" \
  "polars" \ 
  "datasets" \
  "accelerate" \
  "huggingface_hub" \
  "tensorboard" \
  "matplotlib" \
  "tqdm"

# Now, install the correct CUDA-enabled PyTorch version for the V100.
# We reinstall torch and torchvision to ensure the GPU version is used.
!pip install -q torch==2.3.1 torchvision==0.18.1 --index-url https://download.pytorch.org/whl/cu121

print("✅ Dependencies installed successfully.")

In [ ]:
import torch
import polars as pl
import tarfile
from pathlib import Path
from functools import partial
from typing import Tuple

# --- Primary Imports ---
from huggingface_hub import hf_hub_download, login
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter
from torchcodec.decoders import VideoDecoder
from torchcodec.samplers import clips_at_random_indices
from torchvision.transforms import v2
from transformers import VJEPA2ForVideoClassification, VJEPA2VideoProcessor
from tqdm.auto import tqdm

# --- Device and Data Type Configuration for V100 ---
# This will automatically select "cuda" and use float16 for optimized performance.
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if torch.cuda.is_available() else torch.float32

print(f"✅ Using device: {DEVICE}")
print(f"✅ Using data type: {DTYPE}")

In [ ]:
# --- Download and Extract Dataset ---
print("Downloading and extracting dataset...")
fpath = hf_hub_download(
    repo_id="sayakpaul/ucf101-subset",
    filename="UCF101_subset.tar.gz",
    repo_type="dataset"
)
with tarfile.open(fpath) as t:
    t.extractall(".")
print("Dataset extracted successfully.")

# --- Use Polars to Manage Files ---
dataset_root_path = Path("UCF101_subset")
all_video_files = [str(p) for p in dataset_root_path.glob("**/*.avi")]

df = pl.DataFrame({"path": all_video_files}).with_columns(
    pl.col("path").str.split("/").list.get(1).alias("split"),
    pl.col("path").str.split("/").list.get(2).alias("label")
)

# --- Create Label Mappings ---
unique_labels = df["label"].unique().sort().to_list()
label2id = {label: i for i, label in enumerate(unique_labels)}
id2label = {i: label for label, i in label2id.items()}
df = df.with_columns(pl.col("label").map_dict(label2id).alias("label_id"))

print("\n--- Dataset Overview ---")
print(df.head())

# --- Custom PyTorch Dataset Class ---
class PolarsVideoDataset(Dataset):
    def __init__(self, df: pl.DataFrame):
        self.df = df

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int) -> Tuple[VideoDecoder, int]:
        row = self.df.row(idx, named=True)
        decoder = VideoDecoder(row["path"])
        return decoder, row["label_id"]

# --- Create Datasets and DataLoaders ---
train_df = df.filter(pl.col("split") == "train")
val_df = df.filter(pl.col("split") == "val")
test_df = df.filter(pl.col("split") == "test")

train_ds, val_ds, test_ds = PolarsVideoDataset(train_df), PolarsVideoDataset(val_df), PolarsVideoDataset(test_df)

# --- Define Collate Function and Transforms ---
def collate_fn(samples, frames_per_clip, transforms):
    clips, labels = [], []
    for decoder, lbl in samples:
        clip = clips_at_random_indices(
            decoder, num_clips=1, num_frames_per_clip=frames_per_clip, num_indices_between_frames=3
        ).data
        clips.append(clip)
        labels.append(lbl)
    videos = torch.cat(clips, dim=0)
    videos = transforms(videos)
    return videos, torch.tensor(labels)

model_name = "facebook/vjepa2-vitl-fpc16-256-ssv2"
processor = VJEPA2VideoProcessor.from_pretrained(model_name)
CROP_SIZE = (processor.crop_size["height"], processor.crop_size["width"])
FRAMES_PER_CLIP = 16

train_transforms = v2.Compose([
    v2.RandomResizedCrop(CROP_SIZE, antialias=True),
    v2.RandomHorizontalFlip(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=processor.image_mean, std=processor.image_std),
])
eval_transforms = v2.Compose([
    v2.Resize(CROP_SIZE, antialias=True),
    v2.CenterCrop(CROP_SIZE),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=processor.image_mean, std=processor.image_std),
])

batch_size = 8  # A V100 can handle a larger batch size
num_workers = 2

train_loader = DataLoader(
    train_ds, batch_size=batch_size, shuffle=True,
    collate_fn=partial(collate_fn, frames_per_clip=FRAMES_PER_CLIP, transforms=train_transforms),
    num_workers=num_workers, pin_memory=True,
)
val_loader = DataLoader(
    val_ds, batch_size=batch_size, shuffle=False,
    collate_fn=partial(collate_fn, frames_per_clip=FRAMES_PER_CLIP, transforms=eval_transforms),
    num_workers=num_workers, pin_memory=True,
)
print("\n✅ DataLoaders created successfully.")

In [ ]:
# --- Hugging Face Login ---
# You'll need a Hugging Face account and a User Access Token with "write" permissions.
# Get one here: https://huggingface.co/settings/tokens
login()

# --- Model Initialization ---
model = VJEPA2ForVideoClassification.from_pretrained(
    model_name,
    torch_dtype=DTYPE,
    label2id=label2id,
    id2label=id2label,
    ignore_mismatched_sizes=True,
).to(DEVICE)

# --- Freeze Backbone & Set up Optimizer ---
for param in model.vjepa2.parameters():
    param.requires_grad = False

optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-4)
scaler = torch.cuda.amp.GradScaler() # For stable mixed-precision training

# --- Evaluation Function ---
def evaluate(loader, model, processor, device, dtype) -> float:
    model.eval()
    correct, total = 0, 0
    with torch.inference_mode():
        for vids, labels in tqdm(loader, desc="Evaluating", leave=False):
            inputs = processor(vids, return_tensors="pt")
            inputs = {k: v.to(device) for k, v in inputs.items()}
            labels = labels.to(device)
            with torch.autocast(device_type=device, dtype=dtype):
                logits = model(**inputs).logits
            preds = logits.argmax(-1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return correct / total

# --- Training Loop ---
num_epochs = 5
writer = SummaryWriter("runs/vjepa2_finetune_ucf101")
print("\n--- Starting Training ---")

for epoch in range(num_epochs):
    model.train()
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")
    for vids, labels in progress_bar:
        inputs = processor(vids, return_tensors="pt")
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
        labels = labels.to(DEVICE)
        with torch.autocast(device_type=DEVICE, dtype=DTYPE):
            outputs = model(**inputs, labels=labels)
            loss = outputs.loss
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad()
        progress_bar.set_postfix({"loss": f"{loss.item():.4f}"})

    # --- Validation after each epoch ---
    val_acc = evaluate(val_loader, model, processor, DEVICE, DTYPE)
    print(f"Epoch {epoch+1} Validation Accuracy: {val_acc:.4f}")
    writer.add_scalar("Validation Accuracy", val_acc, epoch + 1)

writer.close()
print("\n✅ Training complete.")

In [ ]:
# --- Create Test Loader ---
test_loader = DataLoader(
    test_ds, batch_size=batch_size, shuffle=False,
    collate_fn=partial(collate_fn, frames_per_clip=FRAMES_PER_CLIP, transforms=eval_transforms),
    num_workers=num_workers, pin_memory=True,
)

# --- Final Test Evaluation ---
test_acc = evaluate(test_loader, model, processor, DEVICE, DTYPE)
print(f"✅ Final Test Accuracy: {test_acc:.4f}")

# --- Push to Hub ---
hub_model_id = "dzeng/vjepa2-vitl-ucf101-finetuned"
print(f"Pushing model to {hub_model_id}...")
model.push_to_hub(hub_model_id)
processor.push_to_hub(hub_model_id)
print(f"🚀 Model pushed successfully!")

# --- Inference with Fine-Tuned Model ---
print("\n--- Testing with a new video ---")
video_url = "https://huggingface.co/datasets/merve/vlm_test_images/resolve/main/IMG_3830.mp4"
vr = VideoDecoder(video_url)
indices = torch.linspace(0, len(vr) - 1, FRAMES_PER_CLIP).long()
video_frames = vr.get_frames_at(indices=indices.numpy()).data
inputs = processor([video_frames], return_tensors="pt")
inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

with torch.inference_mode():
    logits = model(**inputs).logits
    predicted_idx = logits.argmax(-1).item()

# The closest label in UCF-101 for a concert video is "BandMarching"
print(f"Predicted class: {model.config.id2label[predicted_idx]}")